# DualEncoderSeg v3 vs UNet Baseline — Comparison Experiment

> **Author**: Anirban | **Model**: DualEncoderSeg v3 (dual-path, latent-space bridging)  
> **Comparison target**: Standard UNet with matched encoder depth & conv style  

---

## Notebook Structure

| Section | Content |
|---|---|
| §0 | Environment & Imports |
| §1 | **Global Config** — change this cell only |
| §2 | Model Specifications (reference table) |
| §3 | Shared Primitive Blocks |
| §4 | DualEncoderSeg v3 Architecture |
| §5 | UNet Baseline (matched depth) |
| §6 | Loss Functions |
| §7 | Metrics |
| §8 | Dataset & DataLoaders |
| §9 | Training Functions |
| §10 | **Run Training** |
| §11 | **Results & Comparison Plots** |
| §12 | Inference Utilities |

> **Tip**: To run a different experiment, only modify **Section 1** and re-run all cells.

## §0 — Environment & Imports

In [ ]:
# Cell 0.1 — Install dependencies (run once)
# Comment out if packages are already installed
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'torchvision', 'Pillow', 'matplotlib',
                'seaborn', 'numpy', 'tqdm'], check=False)

In [ ]:
# Cell 0.2 — Imports
import os, sys, math, random, json, time
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Tuple, Dict
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

matplotlib.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_theme(style='whitegrid', palette='tab10')

print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')

## §1 — Global Config  *(change only this cell)*

In [ ]:
# Cell 1.1 — Experiment Config
# ─────────────────────────────────────────────────────────────────────
# THIS is the single cell you modify to change the entire experiment.
# All other cells read from `CFG`.
# ─────────────────────────────────────────────────────────────────────

@dataclass
class ExperimentConfig:
    # ── Data ────────────────────────────────────────────────────────
    # Set DATA_ROOT to your dataset folder.
    # Expected layout:  DATA_ROOT/train/image/*.png
    #                   DATA_ROOT/train/mask/*.png
    DATA_ROOT:    str   = '/kaggle/input/YOUR_DATASET/Data/train'
    image_size:   int   = 512
    train_split:  float = 0.8     # fraction used for training
    batch_size:   int   = 4
    num_workers:  int   = 2
    seed:         int   = 42

    # ── Device ──────────────────────────────────────────────────────
    device:       str   = 'cuda' if torch.cuda.is_available() else 'cpu'

    # ── Latent / Encoder ────────────────────────────────────────────
    spatial_h:    int   = 16      # latent spatial height (image_size / 2^5)
    spatial_w:    int   = 16      # latent spatial width
    latent_ch:    int   = 128     # latent channel count
    dropout:      float = 0.1

    # ── DualEncoderSeg v3: training schedule ────────────────────────
    stage1_epochs:        int   = 60    # MaskAutoencoder pre-training
    stage2_epochs:        int   = 120   # DualEncoderSegV3 full training
    stage1_lr:            float = 1e-3
    stage2_lr:            float = 3e-4
    warmup_epochs:        int   = 10
    latent_warmup_epochs: int   = 20    # VICReg weight schedule

    # ── UNet Baseline: training schedule ────────────────────────────
    unet_epochs:  int   = 120
    unet_lr:      float = 3e-4

    # ── Loss weights ────────────────────────────────────────────────
    lambda_vicreg_sim: float = 5.0
    lambda_vicreg_var: float = 5.0
    lambda_vicreg_cov: float = 1.0
    lambda_tversky:    float = 1.5
    lambda_boundary:   float = 1.0
    lambda_aux:        float = 0.4
    tversky_alpha:     float = 0.3
    tversky_beta:      float = 0.7

    # ── Checkpoints ─────────────────────────────────────────────────
    save_dir:     str   = './checkpoints'


CFG = ExperimentConfig()

# Seed everything
torch.manual_seed(CFG.seed)
np.random.seed(CFG.seed)
random.seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

os.makedirs(CFG.save_dir, exist_ok=True)
print(f'Device: {CFG.device}  |  Image size: {CFG.image_size}  |  Batch: {CFG.batch_size}')
print(f'Stage1 epochs: {CFG.stage1_epochs}  Stage2 epochs: {CFG.stage2_epochs}  UNet epochs: {CFG.unet_epochs}')

## §2 — Model Specifications  *(reference — do not modify)*

In [ ]:
# Cell 2.1 — Architecture specification table
# Printed at runtime so it reflects the actual CFG values.

spec_rows = [
    ('Spec',                   'DualEncoderSeg v3',                        'UNet Baseline'),
    ('─' * 22,                 '─' * 34,                                   '─' * 30),
    ('Encoder type',           'Custom SEResBlock CNN',                    'Standard DoubleConv CNN'),
    ('Down-sampling',          'Stride-2 Conv  (no MaxPool)',              'MaxPool 2×2'),
    ('Encoder depth',          '5 stages  (512→256→128→64→32→16)',         '5 stages  (512→256→128→64→32→16)'),
    ('Skip mechanism',         'SkipFusionGate  (learned sigmoid gate)',    'Concat → DoubleConv  (fixed)'),
    ('Latent representation',  f'Spatial {CFG.spatial_h}×{CFG.spatial_w}×{CFG.latent_ch}',
                               'Bottleneck 16×16×512'),
    ('Decoder',                'Bilinear upsample + ConvBN',               'Bilinear upsample + DoubleConv'),
    ('Training stages',        '2  (MAE pretrain → cross-modal finetune)', '1  (end-to-end)'),
    ('Loss terms',             'VICReg + Tversky + Boundary + Aux',        'Tversky + Boundary'),
    ('Aux head',               'Yes  (32×32 latent-state prediction)',      'No'),
    ('VICReg latent align',    'Yes  (z_pred ↔ z_mask)',                   'No'),
    ('Stage1 epochs',          str(CFG.stage1_epochs),                     'N/A'),
    ('Stage2/UNet epochs',     str(CFG.stage2_epochs),                     str(CFG.unet_epochs)),
    ('Learning rate',          str(CFG.stage2_lr),                         str(CFG.unet_lr)),
]

col_w = [24, 38, 32]
for row in spec_rows:
    print('  '.join(str(c).ljust(w) for c, w in zip(row, col_w)))

## §3 — Shared Primitive Blocks

In [ ]:
# Cell 3.1 — ConvBNReLU, SEResBlock, DownStage

class ConvBNReLU(nn.Module):
    """Conv2d → BatchNorm → ReLU. The atomic building block."""
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1, groups=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride, padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class SEResBlock(nn.Module):
    """
    Squeeze-and-Excitation Residual Block.
    Adds channel recalibration on top of a standard residual connection.
    Used in: MaskEncoder, ImageEncoder (DualEncoderSeg v3 only).
    """
    def __init__(self, ch, se_ratio=4):
        super().__init__()
        self.conv = nn.Sequential(
            ConvBNReLU(ch, ch),
            nn.Conv2d(ch, ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(ch),
        )
        hidden = max(ch // se_ratio, 4)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, hidden, bias=False), nn.ReLU(inplace=True),
            nn.Linear(hidden, ch, bias=False), nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res   = self.conv(x)
        scale = self.se(res).view(res.size(0), -1, 1, 1)
        return self.relu(x + res * scale)


class DownStage(nn.Module):
    """
    Stride-2 conv + SEResBlock.
    DualEncoderSeg uses stride-2 conv for downsampling instead of MaxPool.
    Advantage: learnable downsampling, no information loss from MaxPool.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNReLU(in_ch, out_ch, stride=2),
            SEResBlock(out_ch),
        )
    def forward(self, x): return self.block(x)


print('Primitive blocks defined: ConvBNReLU, SEResBlock, DownStage')

In [ ]:
# Cell 3.2 — SkipFusionGate (DualEncoderSeg v3 only)

class SkipFusionGate(nn.Module):
    """
    Learned gated fusion of a skip connection into decoder context.

    Problem it solves:
      In Stage 2, skip features come from the IMAGE encoder while the
      decoder was trained on MASK features (different representational
      spaces). A plain concat/add causes destructive interference.

    Gate formula:
      g   = sigmoid(W_gate · cat(skip, ctx))   # per-channel confidence
      out = ctx_proj(ctx) + g ⊙ skip_proj(skip)  # gated residual

    In Stage 1: mask skip and decoder context are aligned → gate opens wide.
    In Stage 2: cross-modal domain gap → gate learns selective trust.
    """
    def __init__(self, skip_ch, ctx_ch, out_ch):
        super().__init__()
        self.skip_proj = nn.Conv2d(skip_ch, out_ch, 1, bias=False)
        self.gate      = nn.Sequential(
            nn.Conv2d(skip_ch + ctx_ch, out_ch, 1, bias=True),
            nn.Sigmoid(),
        )
        self.ctx_proj  = nn.Conv2d(ctx_ch, out_ch, 1, bias=False)
        self.bn        = nn.BatchNorm2d(out_ch)

    def forward(self, skip: torch.Tensor, ctx: torch.Tensor) -> torch.Tensor:
        if skip.shape[-2:] != ctx.shape[-2:]:
            skip = F.interpolate(skip, size=ctx.shape[-2:],
                                 mode='bilinear', align_corners=False)
        gate  = self.gate(torch.cat([skip, ctx], dim=1))
        fused = self.ctx_proj(ctx) + gate * self.skip_proj(skip)
        return self.bn(fused)


print('SkipFusionGate defined')

## §4 — DualEncoderSeg v3 Architecture

In [ ]:
# Cell 4.1 — MaskEncoder & ImageEncoder

class MaskEncoder(nn.Module):
    """
    Encodes 512×512×1 binary mask → latent (B, latent_ch, 16, 16).
    Returns latent z AND skip features [s1, s2, s3, s4].

    Resolution pyramid:
      stem   → 512×512×32
      down1  → 256×256×64   (s1)
      down2  → 128×128×128  (s2)
      down3  →  64×64×256   (s3)
      down4  →  32×32×256   (s4)
      down5  →  16×16×256
      project→  16×16×latent_ch  (z_mask)
    """
    SKIP_CHS = [64, 128, 256, 256]  # s1 … s4 channel counts

    def __init__(self, cfg):
        super().__init__()
        self.stem  = ConvBNReLU(1, 32, stride=1)
        self.down1 = DownStage(32,  64)
        self.down2 = DownStage(64,  128)
        self.down3 = DownStage(128, 256)
        self.down4 = DownStage(256, 256)
        self.down5 = DownStage(256, 256)
        self.project = nn.Sequential(
            nn.Conv2d(256, cfg.latent_ch, 1, bias=False),
            nn.BatchNorm2d(cfg.latent_ch),
        )

    def forward(self, x):
        x  = self.stem(x)
        s1 = self.down1(x)
        s2 = self.down2(s1)
        s3 = self.down3(s2)
        s4 = self.down4(s3)
        x5 = self.down5(s4)
        z  = self.project(x5)
        return z, [s1, s2, s3, s4]


class ImageEncoder(nn.Module):
    """
    Encodes 512×512×3 RGB image → latent (B, latent_ch, 16, 16).
    Mirrors MaskEncoder depth exactly — skip channels are identical.
    Critical: same resolution pyramid so cross-modal skip injection works.

    Differences from MaskEncoder:
      - in_ch = 3 (RGB)
      - stem outputs 64ch (vs 32ch in MaskEncoder)
      - Dropout2d before latent projection
    """
    SKIP_CHS = [64, 128, 256, 256]  # f1 … f4 — must match MaskEncoder.SKIP_CHS

    def __init__(self, cfg):
        super().__init__()
        self.stem  = ConvBNReLU(3, 64, stride=1)
        self.down1 = DownStage(64,  64)
        self.down2 = DownStage(64,  128)
        self.down3 = DownStage(128, 256)
        self.down4 = DownStage(256, 256)
        self.down5 = DownStage(256, 256)
        self.project = nn.Sequential(
            nn.Dropout2d(cfg.dropout),
            nn.Conv2d(256, cfg.latent_ch, 1, bias=False),
            nn.BatchNorm2d(cfg.latent_ch),
        )

    def forward(self, x):
        x  = self.stem(x)
        f1 = self.down1(x)
        f2 = self.down2(f1)
        f3 = self.down3(f2)
        f4 = self.down4(f3)
        x5 = self.down5(f4)
        z  = self.project(x5)
        return z, [f1, f2, f3, f4]


print('MaskEncoder and ImageEncoder defined')

In [ ]:
# Cell 4.2 — MaskDecoder with SkipFusionGate

class MaskDecoder(nn.Module):
    """
    Decodes latent (B, latent_ch, 16, 16) → mask logits (B, 1, 512, 512).

    Skip injection at 4 resolutions (low→high):
      after up1 (32×32)  : s4 / f4
      after up2 (64×64)  : s3 / f3
      after up3 (128×128): s2 / f2
      after up4 (256×256): s1 / f1

    In Stage 1: skips are from MaskEncoder (same domain).
    In Stage 2: skips are from ImageEncoder (cross-modal).
    The SkipFusionGate handles the domain gap gracefully.
    """
    def __init__(self, cfg, skip_chs=None):
        super().__init__()
        if skip_chs is None:
            skip_chs = MaskEncoder.SKIP_CHS
        lch = cfg.latent_ch

        self.expand = ConvBNReLU(lch, 256, kernel=1, padding=0)
        self.up1    = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), ConvBNReLU(256, 256))
        self.gate4  = SkipFusionGate(skip_chs[3], 256, 256)
        self.up2    = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), ConvBNReLU(256, 128))
        self.gate3  = SkipFusionGate(skip_chs[2], 128, 128)
        self.up3    = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), ConvBNReLU(128, 128))
        self.gate2  = SkipFusionGate(skip_chs[1], 128, 128)
        self.up4    = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), ConvBNReLU(128, 64))
        self.gate1  = SkipFusionGate(skip_chs[0], 64, 64)
        self.up5    = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), ConvBNReLU(64, 32))
        self.out    = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, z, skips=None):
        x = self.expand(z)
        x = self.up1(x)
        if skips is not None: x = self.gate4(skips[3], x)
        x = self.up2(x)
        if skips is not None: x = self.gate3(skips[2], x)
        x = self.up3(x)
        if skips is not None: x = self.gate2(skips[1], x)
        x = self.up4(x)
        if skips is not None: x = self.gate1(skips[0], x)
        x = self.up5(x)
        return self.out(x)


print('MaskDecoder defined')

In [ ]:
# Cell 4.3 — MaskAutoencoder (Stage 1 module)

class MaskAutoencoder(nn.Module):
    """
    Stage 1 self-supervised module.
    mask → MaskEncoder → (z, skips) → MaskDecoder(z, skips) → recon
    Trains the decoder to use skip context for thin vessel reconstruction.
    """
    def __init__(self, cfg):
        super().__init__()
        self.encoder = MaskEncoder(cfg)
        self.decoder = MaskDecoder(cfg, skip_chs=MaskEncoder.SKIP_CHS)

    def forward(self, mask):
        z, skips = self.encoder(mask)
        recon    = self.decoder(z, skips)
        return recon, z, skips

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False
        for m in self.encoder.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        print('MaskEncoder frozen (weights + BN stats).')


print('MaskAutoencoder defined')

In [ ]:
# Cell 4.4 — SpatialMappingNet & AuxHead

class SpatialMappingNet(nn.Module):
    """
    Maps image spatial latent → mask spatial latent.
    Both: (B, latent_ch, 16, 16).

    Architecture:
      1×1 conv → channel expansion (per-position MLP)
      3×3 depthwise → spatial context (vessel continuity)
      1×1 conv → project back
      residual connection for gradient stability
    """
    def __init__(self, cfg):
        super().__init__()
        ch = cfg.latent_ch
        hidden = ch * 2
        self.net = nn.Sequential(
            nn.Conv2d(ch, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden), nn.GELU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),  # depthwise
            nn.Conv2d(hidden, hidden, 1, bias=False),                            # pointwise
            nn.BatchNorm2d(hidden), nn.GELU(),
            nn.Dropout2d(cfg.dropout),
            nn.Conv2d(hidden, ch, 1, bias=False),
            nn.BatchNorm2d(ch),
        )
        self.residual = nn.Conv2d(ch, ch, 1, bias=False)

    def forward(self, z):
        return self.net(z) + self.residual(z)


class AuxHead(nn.Module):
    """
    Prediction AT the latent state (your original design intent).
    z_pred (16×16) → upsample → lightweight conv head → logits (32×32).

    Why it matters:
      Without this: gradients flow loss → 5 decoder stages → MappingNet (very long path).
      With this:    direct path aux_loss → MappingNet (stable early training).
    At inference: NOT called → zero overhead.
    """
    def __init__(self, cfg):
        super().__init__()
        ch = cfg.latent_ch
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(ch, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
        )

    def forward(self, z):
        return self.head(z)


print('SpatialMappingNet and AuxHead defined')

In [ ]:
# Cell 4.5 — DualEncoderSegV3 (full model)

class DualEncoderSegV3(nn.Module):
    """
    DualEncoderSeg v3 — Full segmentation model.

    Training forward (mask provided):
      image → ImageEncoder → (z_img, [f1..f4])
      z_img → MappingNet   → z_pred
      z_pred → AuxHead     → aux_logits (32×32, short gradient path)
      z_pred + [f1..f4] → MaskDecoder → pred_logits (512×512)
      mask → MaskEncoder (frozen) → z_mask (for VICReg alignment)

    Inference forward (mask=None):
      image → ImageEncoder → z_img → MappingNet → z_pred
      z_pred + image_skips → MaskDecoder → logits
    """
    def __init__(self, cfg, pretrained_mae=None):
        super().__init__()
        self.cfg           = cfg
        self.image_encoder = ImageEncoder(cfg)
        self.mapping       = SpatialMappingNet(cfg)
        self.aux_head      = AuxHead(cfg)

        if pretrained_mae is not None:
            self.mask_decoder = pretrained_mae.decoder
            self.mask_encoder = pretrained_mae.encoder
            pretrained_mae.freeze_encoder()
        else:
            self.mask_encoder = MaskEncoder(cfg)
            self.mask_decoder = MaskDecoder(cfg, skip_chs=ImageEncoder.SKIP_CHS)

    def forward(self, image, mask=None):
        z_img, img_skips = self.image_encoder(image)
        z_pred           = self.mapping(z_img)
        pred_logits      = self.mask_decoder(z_pred, img_skips)

        if mask is not None:
            aux_logits = self.aux_head(z_pred)
            with torch.no_grad():
                z_mask, _ = self.mask_encoder(mask)
            return pred_logits, aux_logits, z_pred, z_mask

        return pred_logits


def count_params(model):
    total    = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


print('DualEncoderSegV3 defined')

## §5 — UNet Baseline  *(matched depth, standard architecture)*

In [ ]:
# Cell 5.1 — UNet primitive blocks

class DoubleConv(nn.Module):
    """
    Standard UNet building block: Conv3×3 → BN → ReLU → Conv3×3 → BN → ReLU.
    No SE attention, no stride-2 (downsampling handled by MaxPool externally).
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNetDown(nn.Module):
    """MaxPool → DoubleConv. Standard UNet encoder stage."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x): return self.conv(self.pool(x))


class UNetUp(nn.Module):
    """
    Bilinear upsample → concat skip → DoubleConv.
    Standard UNet decoder stage with fixed concat skip (no gating).
    """
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


print('UNet blocks defined: DoubleConv, UNetDown, UNetUp')

In [ ]:
# Cell 5.2 — UNet Baseline model

class UNetBaseline(nn.Module):
    """
    Standard UNet with 5 encoder stages.
    Matched to DualEncoderSegV3 in:
      - Input/output resolution (512×512)
      - Encoder depth (5 stages)
      - Similar parameter budget

    Key differences from DualEncoderSegV3:
      - MaxPool downsampling (vs stride-2 conv in DualEnc)
      - DoubleConv blocks (vs SEResBlock in DualEnc)
      - Concat+DoubleConv skip fusion (vs learned SkipFusionGate)
      - Single training stage (vs Stage1+Stage2)
      - No latent alignment loss

    Encoder channel progression: 3 → 64 → 128 → 256 → 512 → 512
    Decoder channel progression: 512 → 256 → 128 → 64 → 32 → 1
    """
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        # ── Encoder ─────────────────────────────────────────────────
        self.enc1 = DoubleConv(in_ch, 64)     # 512×512×64  (skip e1)
        self.enc2 = UNetDown(64,  128)         # 256×256×128 (skip e2)
        self.enc3 = UNetDown(128, 256)         # 128×128×256 (skip e3)
        self.enc4 = UNetDown(256, 512)         #  64×64×512  (skip e4)
        # ── Bottleneck ──────────────────────────────────────────────
        self.bottleneck = UNetDown(512, 512)   #  32×32×512  (matches 16→32 in DualEnc)
        # ── Decoder ─────────────────────────────────────────────────
        self.dec4 = UNetUp(512, 512, 256)      #  64×64×256
        self.dec3 = UNetUp(256, 256, 128)      # 128×128×128
        self.dec2 = UNetUp(128, 128, 64)       # 256×256×64
        self.dec1 = UNetUp(64,   64, 32)       # 512×512×32
        # ── Output ──────────────────────────────────────────────────
        self.out  = nn.Conv2d(32, out_ch, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bottleneck(e4)
        # Decoder
        d4 = self.dec4(b,  e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.out(d1)


print('UNetBaseline defined')

In [ ]:
# Cell 5.3 — Parameter count comparison

_dual = DualEncoderSegV3(CFG)
_unet = UNetBaseline()

dual_total, dual_train = count_params(_dual)
unet_total, unet_train = count_params(_unet)

del _dual, _unet  # free memory

print(f'{'Model':<24} {'Total Params':>14}  {'Trainable':>14}')
print('-' * 55)
print(f'{'DualEncoderSeg v3':<24} {dual_total:>14,}  {dual_train:>14,}')
print(f'{'UNet Baseline':<24} {unet_total:>14,}  {unet_train:>14,}')
print(f'\nNote: DualEncoderSeg v3 Stage 2 freezes MaskEncoder,')
print(f'      so effective trainable params at Stage 2 are lower.')

## §6 — Loss Functions

In [ ]:
# Cell 6.1 — Shared pixel-level losses

def tversky_loss(pred_logits, target, alpha=0.3, beta=0.7, smooth=1e-5):
    """
    Asymmetric Dice (Tversky). beta > alpha → missing vessels penalised more.
    Applied identically to both DualEncoderSeg v3 and UNet.
    """
    pred = torch.sigmoid(pred_logits)
    tp   = (pred * target).sum(dim=(2, 3))
    fp   = (pred * (1 - target)).sum(dim=(2, 3))
    fn   = ((1 - pred) * target).sum(dim=(2, 3))
    tv   = (tp + smooth) / (tp + alpha * fp + beta * fn + smooth)
    return (1 - tv).mean()


def boundary_loss(pred_logits, target):
    """
    Boundary-weighted BCE using Sobel edge detection on GT mask.
    Applies 5× weight at vessel edges.
    Applied identically to both DualEncoderSeg v3 and UNet.
    """
    sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]],
                            device=target.device).view(1, 1, 3, 3)
    sobel_y = sobel_x.transpose(2, 3)
    edge    = (F.conv2d(target, sobel_x, padding=1).abs() +
               F.conv2d(target, sobel_y, padding=1).abs()).clamp(0, 1)
    bce      = F.binary_cross_entropy_with_logits(pred_logits, target, reduction='none')
    return (bce * (1 + 4 * edge)).mean()


def vicreg_loss(z_pred, z_mask, sim_w=5.0, var_w=5.0, cov_w=1.0):
    """
    VICReg — latent alignment loss for DualEncoderSeg v3 only.
    Variance/covariance computed over batch B (NOT B*H*W — that was the v2 bug).
    """
    B, C, H, W = z_pred.shape
    sim   = F.mse_loss(z_pred, z_mask.detach())
    std_p = torch.sqrt(z_pred.var(dim=0) + 1e-4)
    std_m = torch.sqrt(z_mask.var(dim=0) + 1e-4)
    var   = F.relu(1 - std_p).mean() + F.relu(1 - std_m).mean()

    def _off_diag_cov(z):
        zp  = z - z.mean(dim=0, keepdim=True)
        zf  = zp.permute(2, 3, 0, 1).reshape(H * W, B, C)
        cov = torch.bmm(zf.transpose(1, 2), zf) / max(B - 1, 1)
        mask_eye = 1 - torch.eye(C, device=z.device).unsqueeze(0)
        return (cov.pow(2) * mask_eye).sum() / (C * H * W)

    cov = _off_diag_cov(z_pred) + _off_diag_cov(z_mask)
    return sim_w * sim + var_w * var + cov_w * cov


print('Loss functions defined: tversky_loss, boundary_loss, vicreg_loss')

In [ ]:
# Cell 6.2 & 6.3 — Loss modules for each model

class SegmentationLoss(nn.Module):
    """DualEncoderSeg v3 combined Stage 2 loss."""
    def __init__(self, cfg, lambda_vicreg=1.0):
        super().__init__()
        self.cfg = cfg
        self.lv  = lambda_vicreg

    def forward(self, pred_logits, aux_logits, z_pred, z_mask, target):
        l_vic = vicreg_loss(z_pred, z_mask,
                            sim_w=self.cfg.lambda_vicreg_sim,
                            var_w=self.cfg.lambda_vicreg_var,
                            cov_w=self.cfg.lambda_vicreg_cov)
        l_tvk = tversky_loss(pred_logits, target, self.cfg.tversky_alpha, self.cfg.tversky_beta)
        l_bnd = boundary_loss(pred_logits, target)
        aux_target = F.interpolate(target, size=(32, 32), mode='bilinear', align_corners=False)
        l_aux = F.binary_cross_entropy_with_logits(aux_logits, aux_target)
        total = (self.lv * l_vic
                 + self.cfg.lambda_tversky  * l_tvk
                 + self.cfg.lambda_boundary * l_bnd
                 + self.cfg.lambda_aux      * l_aux)
        breakdown = {'vicreg': l_vic.item(), 'tversky': l_tvk.item(),
                     'boundary': l_bnd.item(), 'aux': l_aux.item()}
        return total, breakdown


class AutoencoderLoss(nn.Module):
    """Stage 1 loss: Tversky + Boundary on mask reconstruction."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    def forward(self, recon_logits, target):
        return (tversky_loss(recon_logits, target,
                             alpha=self.cfg.tversky_alpha, beta=self.cfg.tversky_beta)
                + boundary_loss(recon_logits, target))


class UNetLoss(nn.Module):
    """
    UNet loss: same pixel-level losses as DualEncoderSeg v3 Stage 2.
    No VICReg, no Aux. Fair comparison at pixel level.
    """
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    def forward(self, pred_logits, target):
        l_tvk = tversky_loss(pred_logits, target, self.cfg.tversky_alpha, self.cfg.tversky_beta)
        l_bnd = boundary_loss(pred_logits, target)
        total = self.cfg.lambda_tversky * l_tvk + self.cfg.lambda_boundary * l_bnd
        breakdown = {'tversky': l_tvk.item(), 'boundary': l_bnd.item()}
        return total, breakdown


print('Loss modules defined: SegmentationLoss, AutoencoderLoss, UNetLoss')

## §7 — Metrics

In [ ]:
# Cell 7.1 — compute_metrics

@torch.no_grad()
def compute_metrics(pred_logits, target, threshold=0.5):
    """
    Returns dict of: iou, dice, accuracy, sensitivity, specificity.
    Works for both models — same function, fair comparison.
    """
    pred = (torch.sigmoid(pred_logits) > threshold).float()
    tp = (pred * target).sum(dim=(1, 2, 3))
    fp = (pred * (1 - target)).sum(dim=(1, 2, 3))
    fn = ((1 - pred) * target).sum(dim=(1, 2, 3))
    tn = ((1 - pred) * (1 - target)).sum(dim=(1, 2, 3))
    s  = 1e-5
    return {
        'iou':         ((tp + s) / (tp + fp + fn + s)).mean().item(),
        'dice':        ((2 * tp + s) / (2 * tp + fp + fn + s)).mean().item(),
        'acc':         (pred == target).float().mean().item(),
        'sensitivity': ((tp + s) / (tp + fn + s)).mean().item(),
        'specificity': ((tn + s) / (tn + fp + s)).mean().item(),
    }


print('compute_metrics defined  (IoU, Dice, Accuracy, Sensitivity, Specificity)')

## §8 — Dataset & DataLoaders

In [ ]:
# Cell 8.1 — RetinaDataset

class RetinaDataset(Dataset):
    """
    Retina segmentation dataset.
    Spatial augmentations applied identically to image + mask.
    Photometric augmentations applied to image only.
    """
    def __init__(self, image_paths, mask_paths, augment=False, image_size=512):
        assert len(image_paths) == len(mask_paths), 'Image/mask count mismatch'
        self.image_paths = list(image_paths)
        self.mask_paths  = list(mask_paths)
        self.augment     = augment
        self.size        = image_size

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        from PIL import Image
        import torchvision.transforms.functional as TF

        img  = Image.open(self.image_paths[idx]).convert('RGB').resize((self.size, self.size))
        mask = Image.open(self.mask_paths[idx]).convert('L').resize((self.size, self.size))

        if self.augment:
            if random.random() > 0.5: img, mask = TF.hflip(img), TF.hflip(mask)
            if random.random() > 0.5: img, mask = TF.vflip(img), TF.vflip(mask)
            angle = random.uniform(-30, 30)
            img   = TF.rotate(img,  angle, fill=0)
            mask  = TF.rotate(mask, angle, fill=0)
            img   = TF.adjust_brightness(img, random.uniform(0.7, 1.3))
            img   = TF.adjust_contrast(img,   random.uniform(0.7, 1.3))
            img   = TF.adjust_saturation(img, random.uniform(0.7, 1.3))
            if random.random() > 0.7: img = TF.gaussian_blur(img, kernel_size=3)

        img  = TF.to_tensor(img)           # 3×H×W, [0,1]
        mask = TF.to_tensor(mask)          # 1×H×W, [0,1]
        mask = (mask > 0.5).float()        # binarise
        return img, mask


print('RetinaDataset defined')

In [ ]:
# Cell 8.2 — Instantiate DataLoaders
# Modify DATA_ROOT in Cell 1.1 (CFG.DATA_ROOT) before running this.

data_root = Path(CFG.DATA_ROOT)
img_paths  = sorted((data_root / 'image').glob('*.png'))
mask_paths = sorted((data_root / 'mask').glob('*.png'))

assert len(img_paths) > 0, f'No images found at {data_root / "image"}'
assert len(img_paths) == len(mask_paths), 'Image/mask count mismatch'

split      = int(CFG.train_split * len(img_paths))
train_imgs,  val_imgs  = img_paths[:split],  img_paths[split:]
train_masks, val_masks = mask_paths[:split], mask_paths[split:]

train_dl = DataLoader(
    RetinaDataset(train_imgs, train_masks, augment=True, image_size=CFG.image_size),
    batch_size=CFG.batch_size, shuffle=True,  num_workers=CFG.num_workers, pin_memory=True)

val_dl = DataLoader(
    RetinaDataset(val_imgs, val_masks, augment=False, image_size=CFG.image_size),
    batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)

print(f'Train samples: {len(train_imgs)}  |  Val samples: {len(val_imgs)}')
print(f'Train batches: {len(train_dl)}    |  Val batches: {len(val_dl)}')

## §9 — Training Functions

In [ ]:
# Cell 9.4 — Learning rate scheduler

def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    """Linear warmup → cosine decay."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


print('Scheduler defined')

In [ ]:
# Cell 9.1 — train_stage1: Mask Autoencoder pre-training

def train_stage1(mae, train_loader, val_loader=None, cfg=None, device='cuda'):
    """
    Pretrain MaskAutoencoder (self-supervised mask reconstruction).
    This is DualEncoderSeg-specific — UNet does not use this stage.
    Target: val Dice > 0.88 before proceeding to Stage 2.
    """
    if cfg is None: cfg = CFG
    mae       = mae.to(device)
    criterion = AutoencoderLoss(cfg)
    optimizer = AdamW(mae.parameters(), lr=cfg.stage1_lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.stage1_epochs)
    history   = {'train_loss': [], 'val_dice': [], 'val_iou': [], 'val_sens': []}
    best_dice = 0.0

    print('=' * 60)
    print('STAGE 1 — Mask Autoencoder (skip-aware self-supervised)')
    print('=' * 60)

    pbar = tqdm(range(1, cfg.stage1_epochs + 1), desc='Stage 1')
    for epoch in pbar:
        mae.train()
        total_loss = 0.0
        for batch in train_loader:
            masks = (batch[1] if isinstance(batch, (list, tuple)) else batch).float().to(device)
            optimizer.zero_grad()
            recon, _, _ = mae(masks)
            loss = criterion(recon, masks)
            loss.backward()
            nn.utils.clip_grad_norm_(mae.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        avg = total_loss / len(train_loader)
        history['train_loss'].append(avg)

        if val_loader is not None:
            mae.eval()
            dl, il, sl = [], [], []
            with torch.no_grad():
                for batch in val_loader:
                    masks = (batch[1] if isinstance(batch, (list, tuple)) else batch).float().to(device)
                    recon, _, _ = mae(masks)
                    m = compute_metrics(recon, masks)
                    dl.append(m['dice']); il.append(m['iou']); sl.append(m['sensitivity'])
            vd, vi, vs = np.mean(dl), np.mean(il), np.mean(sl)
            history['val_dice'].append(vd)
            history['val_iou'].append(vi)
            history['val_sens'].append(vs)
            best_dice = max(best_dice, vd)
            pbar.set_postfix({'loss': f'{avg:.4f}', 'Dice': f'{vd:.4f}', 'IoU': f'{vi:.4f}'})

    if best_dice < 0.88 and val_loader is not None:
        print(f'WARNING: best val Dice={best_dice:.4f} < 0.88 — consider more epochs.')
    print(f'Stage 1 complete. Best Dice: {best_dice:.4f}\n')
    return history


print('train_stage1 defined')

In [ ]:
# Cell 9.2 — train_stage2: DualEncoderSegV3 full training

def train_stage2(model, train_loader, val_loader=None, cfg=None, device='cuda'):
    """
    Train DualEncoderSegV3. MaskEncoder is frozen.
    VICReg curriculum: high weight for first latent_warmup_epochs, then lower.
    """
    if cfg is None: cfg = CFG
    model     = model.to(device)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=cfg.stage2_lr, weight_decay=1e-4)
    scheduler = get_warmup_cosine_scheduler(optimizer, cfg.warmup_epochs, cfg.stage2_epochs)
    history   = {'train_loss': [], 'val_dice': [], 'val_iou': [], 'val_sens': [],
                 'vicreg': [], 'tversky': [], 'boundary': [], 'aux': []}

    print('=' * 60)
    print('STAGE 2 — DualEncoderSeg v3 (cross-modal skip injection)')
    print(f'Trainable: {sum(p.numel() for p in trainable):,}  '
          f'Frozen: {sum(p.numel() for p in model.parameters() if not p.requires_grad):,}')
    print('=' * 60)

    pbar = tqdm(range(1, cfg.stage2_epochs + 1), desc='Stage 2')
    for epoch in pbar:
        lv = 2.0 if epoch <= cfg.latent_warmup_epochs else 0.5
        criterion = SegmentationLoss(cfg, lambda_vicreg=lv)
        model.train()
        model.mask_encoder.eval()
        for m in model.mask_encoder.modules():
            if isinstance(m, nn.BatchNorm2d): m.eval()

        total_loss = 0.0
        bd_acc = {'vicreg': 0., 'tversky': 0., 'boundary': 0., 'aux': 0.}
        for images, masks in train_loader:
            images = images.float().to(device)
            masks  = masks.float().to(device)
            optimizer.zero_grad()
            pred_logits, aux_logits, z_pred, z_mask = model(images, masks)
            loss, bd = criterion(pred_logits, aux_logits, z_pred, z_mask, masks)
            loss.backward()
            nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            total_loss += loss.item()
            for k in bd_acc: bd_acc[k] += bd[k]

        scheduler.step()
        avg = total_loss / len(train_loader)
        history['train_loss'].append(avg)
        for k in bd_acc: history[k].append(bd_acc[k] / len(train_loader))

        if val_loader is not None:
            model.eval()
            dl, il, sl = [], [], []
            with torch.no_grad():
                for images, masks in val_loader:
                    images = images.float().to(device)
                    masks  = masks.float().to(device)
                    pred   = model(images)
                    m      = compute_metrics(pred, masks)
                    dl.append(m['dice']); il.append(m['iou']); sl.append(m['sensitivity'])
            vd, vi, vs = np.mean(dl), np.mean(il), np.mean(sl)
            history['val_dice'].append(vd)
            history['val_iou'].append(vi)
            history['val_sens'].append(vs)
            pbar.set_postfix({'loss': f'{avg:.4f}', 'Dice': f'{vd:.4f}', 'IoU': f'{vi:.4f}', 'lv': lv})

    print('Stage 2 complete.\n')
    return history


print('train_stage2 defined')

In [ ]:
# Cell 9.3 — train_unet: UNet baseline single-stage training

def train_unet(model, train_loader, val_loader=None, cfg=None, device='cuda'):
    """
    Train UNet Baseline. Single-stage end-to-end.
    Uses the same pixel-level losses as DualEncoderSeg v3 Stage 2.
    """
    if cfg is None: cfg = CFG
    model     = model.to(device)
    criterion = UNetLoss(cfg)
    optimizer = AdamW(model.parameters(), lr=cfg.unet_lr, weight_decay=1e-4)
    scheduler = get_warmup_cosine_scheduler(optimizer, cfg.warmup_epochs, cfg.unet_epochs)
    history   = {'train_loss': [], 'val_dice': [], 'val_iou': [], 'val_sens': [],
                 'tversky': [], 'boundary': []}

    print('=' * 60)
    print('UNet Baseline — single-stage end-to-end training')
    total_p = sum(p.numel() for p in model.parameters())
    print(f'Total params: {total_p:,}')
    print('=' * 60)

    pbar = tqdm(range(1, cfg.unet_epochs + 1), desc='UNet')
    for epoch in pbar:
        model.train()
        total_loss = 0.0
        bd_acc = {'tversky': 0., 'boundary': 0.}
        for images, masks in train_loader:
            images = images.float().to(device)
            masks  = masks.float().to(device)
            optimizer.zero_grad()
            pred   = model(images)
            loss, bd = criterion(pred, masks)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            for k in bd_acc: bd_acc[k] += bd[k]

        scheduler.step()
        avg = total_loss / len(train_loader)
        history['train_loss'].append(avg)
        for k in bd_acc: history[k].append(bd_acc[k] / len(train_loader))

        if val_loader is not None:
            model.eval()
            dl, il, sl = [], [], []
            with torch.no_grad():
                for images, masks in val_loader:
                    images = images.float().to(device)
                    masks  = masks.float().to(device)
                    pred   = model(images)
                    m      = compute_metrics(pred, masks)
                    dl.append(m['dice']); il.append(m['iou']); sl.append(m['sensitivity'])
            vd, vi, vs = np.mean(dl), np.mean(il), np.mean(sl)
            history['val_dice'].append(vd)
            history['val_iou'].append(vi)
            history['val_sens'].append(vs)
            pbar.set_postfix({'loss': f'{avg:.4f}', 'Dice': f'{vd:.4f}', 'IoU': f'{vi:.4f}'})

    print('UNet training complete.\n')
    return history


print('train_unet defined')

## §10 — Run Training

In [ ]:
# Cell 10.1 — Train Stage 1: Mask Autoencoder
# Skip this cell if you have a saved checkpoint.

mae   = MaskAutoencoder(CFG)
hist1 = train_stage1(mae, train_dl, val_dl, cfg=CFG, device=CFG.device)

In [ ]:
# Cell 10.2 — Train Stage 2: DualEncoderSegV3

dual_model = DualEncoderSegV3(CFG, pretrained_mae=mae)
hist2      = train_stage2(dual_model, train_dl, val_dl, cfg=CFG, device=CFG.device)

In [ ]:
# Cell 10.3 — Train UNet Baseline

unet_model = UNetBaseline()
hist_unet  = train_unet(unet_model, train_dl, val_dl, cfg=CFG, device=CFG.device)

In [ ]:
# Cell 10.4 — Save checkpoints

torch.save({'model_state': dual_model.state_dict(), 'cfg': asdict(CFG),
            'hist1': hist1, 'hist2': hist2},
           f'{CFG.save_dir}/dual_encoder_seg_v3.pt')

torch.save({'model_state': unet_model.state_dict(), 'cfg': asdict(CFG),
            'hist_unet': hist_unet},
           f'{CFG.save_dir}/unet_baseline.pt')

print(f'Checkpoints saved to: {CFG.save_dir}/')
print(f'  dual_encoder_seg_v3.pt')
print(f'  unet_baseline.pt')

## §11 — Results & Comparison Plots

In [ ]:
# Cell 11.0 — Plot helper utilities

DUAL_COLOR = '#2196F3'   # blue
UNET_COLOR = '#FF5722'   # deep orange
S1_COLOR   = '#4CAF50'   # green (Stage 1)

def smooth(vals, w=5):
    """Simple moving average for smoother loss curves."""
    if len(vals) < w: return vals
    kernel = np.ones(w) / w
    padded = np.pad(vals, (w//2, w//2), mode='edge')
    return np.convolve(padded, kernel, mode='valid')[:len(vals)]


def final_metrics_from_history(history):
    """Extract final epoch val metrics from history dict."""
    return {k: history[k][-1] for k in ['val_dice', 'val_iou', 'val_sens']
            if k in history and len(history[k]) > 0}


print('Plot helpers defined')

In [ ]:
# Cell 11.1 — Full validation metrics on val set

def evaluate_model(model, val_loader, device, model_name):
    model.eval()
    all_m = {k: [] for k in ['iou', 'dice', 'acc', 'sensitivity', 'specificity']}
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.float().to(device)
            masks  = masks.float().to(device)
            pred   = model(images) if not isinstance(model, DualEncoderSegV3) else model(images)
            m      = compute_metrics(pred, masks)
            for k in all_m: all_m[k].append(m[k])
    means = {k: np.mean(v) for k, v in all_m.items()}
    print(f'\n{model_name} — Final Validation Metrics')
    print('-' * 50)
    for k, v in means.items():
        print(f'  {k:<16}: {v:.4f}')
    return means


dual_final = evaluate_model(dual_model, val_dl, CFG.device, 'DualEncoderSeg v3')
unet_final = evaluate_model(unet_model, val_dl, CFG.device, 'UNet Baseline')

print('\n=== HEAD-TO-HEAD SUMMARY ===')
print(f'{'Metric':<18} {'DualEnc v3':>12}  {'UNet':>10}  {'Delta':>10}')
print('-' * 55)
for k in dual_final:
    delta = dual_final[k] - unet_final[k]
    sign  = '+' if delta >= 0 else ''
    print(f'  {k:<16} {dual_final[k]:>12.4f}  {unet_final[k]:>10.4f}  {sign}{delta:>9.4f}')

In [ ]:
# Cell 11.2 — Training loss curves side-by-side

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Loss Curves', fontsize=14, fontweight='bold', y=1.02)

# DualEncoderSeg v3 Stage 2
ax = axes[0]
ax.set_title('DualEncoderSeg v3  (Stage 2)', fontsize=12)
epochs_d = range(1, len(hist2['train_loss']) + 1)
ax.plot(epochs_d, smooth(hist2['train_loss']), color=DUAL_COLOR, lw=2, label='Train Loss')
ax.axvline(CFG.latent_warmup_epochs, color='gray', linestyle='--', alpha=0.6, label='VICReg warmup end')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# UNet
ax = axes[1]
ax.set_title('UNet Baseline', fontsize=12)
epochs_u = range(1, len(hist_unet['train_loss']) + 1)
ax.plot(epochs_u, smooth(hist_unet['train_loss']), color=UNET_COLOR, lw=2, label='Train Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/plot_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.3 — Val Dice & IoU over epochs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Validation Metrics over Epochs', fontsize=14, fontweight='bold', y=1.02)

for ax, metric, title in zip(axes,
                               ['val_dice', 'val_iou'],
                               ['Validation Dice', 'Validation IoU']):
    if metric in hist2 and hist2[metric]:
        ax.plot(range(1, len(hist2[metric]) + 1), hist2[metric],
                color=DUAL_COLOR, lw=2, label='DualEncoderSeg v3')
    if metric in hist_unet and hist_unet[metric]:
        ax.plot(range(1, len(hist_unet[metric]) + 1), hist_unet[metric],
                color=UNET_COLOR, lw=2, linestyle='--', label='UNet Baseline')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch'); ax.set_ylabel(metric.replace('val_', '').title())
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/plot_metric_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.4 — DualEncoderSeg v3 loss breakdown (stacked area)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Loss Term Breakdown over Training', fontsize=14, fontweight='bold', y=1.02)

# DualEnc breakdown
ax = axes[0]
ax.set_title('DualEncoderSeg v3 — Loss Components', fontsize=11)
keys   = ['vicreg', 'tversky', 'boundary', 'aux']
colors = ['#7986CB', '#EF5350', '#66BB6A', '#FFA726']
epochs = range(1, len(hist2['train_loss']) + 1)
stacks = [smooth(hist2[k]) if k in hist2 and hist2[k] else np.zeros(len(hist2['train_loss']))
          for k in keys]
ax.stackplot(epochs, stacks, labels=['VICReg', 'Tversky', 'Boundary', 'Aux'], colors=colors, alpha=0.8)
ax.axvline(CFG.latent_warmup_epochs, color='black', linestyle='--', alpha=0.5, label='VICReg warmup end')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(loc='upper right', fontsize=8); ax.grid(True, alpha=0.2)

# UNet breakdown
ax = axes[1]
ax.set_title('UNet Baseline — Loss Components', fontsize=11)
keys_u   = ['tversky', 'boundary']
colors_u = ['#EF5350', '#66BB6A']
epochs_u = range(1, len(hist_unet['train_loss']) + 1)
stacks_u = [smooth(hist_unet[k]) if k in hist_unet and hist_unet[k] else np.zeros(len(hist_unet['train_loss']))
            for k in keys_u]
ax.stackplot(epochs_u, stacks_u, labels=['Tversky', 'Boundary'], colors=colors_u, alpha=0.8)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(loc='upper right', fontsize=8); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/plot_loss_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.5 — Final metric grouped bar chart

metrics_labels = ['IoU', 'Dice', 'Accuracy', 'Sensitivity', 'Specificity']
keys_in_dict   = ['iou', 'dice', 'acc', 'sensitivity', 'specificity']

dual_vals = [dual_final[k] for k in keys_in_dict]
unet_vals = [unet_final[k] for k in keys_in_dict]

x     = np.arange(len(metrics_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - width/2, dual_vals, width, label='DualEncoderSeg v3',
            color=DUAL_COLOR, alpha=0.88, edgecolor='white', linewidth=0.5)
b2 = ax.bar(x + width/2, unet_vals, width, label='UNet Baseline',
            color=UNET_COLOR, alpha=0.88, edgecolor='white', linewidth=0.5)

for bar in [*b1, *b2]:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., h + 0.003, f'{h:.3f}',
            ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_ylim(0, 1.12)
ax.set_xticks(x); ax.set_xticklabels(metrics_labels, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Final Validation Metrics — DualEncoderSeg v3 vs UNet Baseline',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/plot_metric_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.6 — Radar chart (spider plot)

categories = ['IoU', 'Dice', 'Accuracy', 'Sensitivity', 'Specificity']
N          = len(categories)
angles     = [n / float(N) * 2 * math.pi for n in range(N)]
angles    += angles[:1]  # close the polygon

dual_r = [dual_final[k] for k in keys_in_dict] + [dual_final[keys_in_dict[0]]]
unet_r = [unet_final[k] for k in keys_in_dict] + [unet_final[keys_in_dict[0]]]

fig, ax = plt.subplots(1, 1, figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(math.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=7, color='grey')

ax.plot(angles, dual_r, color=DUAL_COLOR, lw=2, linestyle='solid')
ax.fill(angles, dual_r, color=DUAL_COLOR, alpha=0.25)
ax.plot(angles, unet_r, color=UNET_COLOR, lw=2, linestyle='dashed')
ax.fill(angles, unet_r, color=UNET_COLOR, alpha=0.15)

ax.set_title('5-Metric Radar Comparison', size=13, fontweight='bold', pad=20)
legend_patches = [
    mpatches.Patch(color=DUAL_COLOR, alpha=0.7, label='DualEncoderSeg v3'),
    mpatches.Patch(color=UNET_COLOR, alpha=0.7, label='UNet Baseline'),
]
ax.legend(handles=legend_patches, loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/plot_radar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.7 — Visual prediction comparison grid
# Rows: Image | GT Mask | DualEnc Prediction | UNet Prediction

N_SAMPLES = 4  # number of validation samples to visualise

dual_model.eval(); unet_model.eval()
sample_imgs, sample_masks = [], []
for imgs, masks in val_dl:
    sample_imgs.append(imgs[:N_SAMPLES])
    sample_masks.append(masks[:N_SAMPLES])
    if len(sample_imgs) * CFG.batch_size >= N_SAMPLES: break

s_imgs  = torch.cat(sample_imgs, 0)[:N_SAMPLES].float().to(CFG.device)
s_masks = torch.cat(sample_masks, 0)[:N_SAMPLES].float().to(CFG.device)

with torch.no_grad():
    dual_preds = torch.sigmoid(dual_model(s_imgs)).cpu().squeeze(1).numpy()
    unet_preds = torch.sigmoid(unet_model(s_imgs)).cpu().squeeze(1).numpy()

s_imgs_np  = s_imgs.cpu().permute(0, 2, 3, 1).numpy()
s_masks_np = s_masks.cpu().squeeze(1).numpy()

fig, axes = plt.subplots(4, N_SAMPLES, figsize=(4 * N_SAMPLES, 14))
fig.suptitle('Visual Comparison — DualEncoderSeg v3 vs UNet Baseline',
             fontsize=14, fontweight='bold', y=1.02)

row_labels = ['Input Image', 'Ground Truth', 'DualEncoderSeg v3', 'UNet Baseline']
data_rows  = [s_imgs_np, s_masks_np, dual_preds > 0.5, unet_preds > 0.5]
cmaps      = ['viridis', 'gray', 'Blues', 'Oranges']
border_cols= [None, 'black', DUAL_COLOR, UNET_COLOR]

for r, (data, rlabel, cmap, bcolor) in enumerate(zip(data_rows, row_labels, cmaps, border_cols)):
    for c in range(N_SAMPLES):
        ax = axes[r, c]
        if data[c].ndim == 3:
            ax.imshow(data[c])
        else:
            ax.imshow(data[c], cmap=cmap, vmin=0, vmax=1)
        ax.axis('off')
        if c == 0:
            ax.set_ylabel(rlabel, fontsize=10, fontweight='bold', rotation=90, labelpad=8)
        if bcolor:
            for spine in ax.spines.values():
                spine.set_edgecolor(bcolor); spine.set_linewidth(2); spine.set_visible(True)

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/plot_visual_grid.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 11.8 — Threshold sweep: Dice vs threshold for both models

thresholds = np.linspace(0.1, 0.9, 17)

def threshold_sweep(model, val_loader, device, thresholds):
    model.eval()
    all_logits, all_masks = [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            logits = model(imgs.float().to(device))
            all_logits.append(logits.cpu())
            all_masks.append(masks.cpu())
    all_logits = torch.cat(all_logits, 0)
    all_masks  = torch.cat(all_masks,  0)
    dice_scores = []
    for t in thresholds:
        m = compute_metrics(all_logits, all_masks, threshold=float(t))
        dice_scores.append(m['dice'])
    return dice_scores


print('Running threshold sweep...')
dual_dice_sweep = threshold_sweep(dual_model, val_dl, CFG.device, thresholds)
unet_dice_sweep = threshold_sweep(unet_model, val_dl, CFG.device, thresholds)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, dual_dice_sweep, 'o-', color=DUAL_COLOR, lw=2,
        markersize=5, label='DualEncoderSeg v3')
ax.plot(thresholds, unet_dice_sweep, 's--', color=UNET_COLOR, lw=2,
        markersize=5, label='UNet Baseline')
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.6, label='Default threshold (0.5)')
ax.set_xlabel('Decision Threshold', fontsize=11)
ax.set_ylabel('Dice Score', fontsize=11)
ax.set_title('Dice Score vs Decision Threshold', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

best_t_dual = thresholds[np.argmax(dual_dice_sweep)]
best_t_unet = thresholds[np.argmax(unet_dice_sweep)]
print(f'Best threshold — DualEnc: {best_t_dual:.2f}  |  UNet: {best_t_unet:.2f}')

plt.tight_layout()
plt.savefig(f'{CFG.save_dir}/plot_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## §12 — Inference Utilities

In [ ]:
# Cell 12.1 — predict_single and predict_batch

@torch.no_grad()
def predict_single(model, image_tensor, threshold=0.5, device=None):
    """
    Predict binary mask for a single image tensor (C×H×W).
    Works for both DualEncoderSegV3 and UNetBaseline.
    Returns: binary mask (H×W) as uint8 tensor.
    """
    if device is None: device = CFG.device
    model.eval()
    x      = image_tensor.unsqueeze(0).float().to(device)
    logits = model(x)
    binary = (torch.sigmoid(logits) > threshold).squeeze().cpu().to(torch.uint8)
    return binary


@torch.no_grad()
def predict_batch(model, images_tensor, threshold=0.5, device=None):
    """
    Predict binary masks for a batch of images (B×C×H×W).
    Returns: binary masks (B×H×W) as uint8 tensor.
    """
    if device is None: device = CFG.device
    model.eval()
    logits = model(images_tensor.float().to(device))
    return (torch.sigmoid(logits) > threshold).squeeze(1).cpu().to(torch.uint8)


print('Inference utilities defined: predict_single, predict_batch')
print('Example usage:')
print('  mask = predict_single(dual_model, image_tensor)   # DualEncoderSeg v3')
print('  mask = predict_single(unet_model, image_tensor)   # UNet Baseline')

In [ ]:
# Cell 12.2 — Final summary of all saved plots

save_dir = Path(CFG.save_dir)
plots    = list(save_dir.glob('plot_*.png'))

print(f'All comparison plots saved to: {save_dir.resolve()}')
print()
for p in sorted(plots):
    print(f'  {p.name}')

print()
print('Plots generated:')
print('  plot_loss_curves.png      — Training loss (Stage2 DualEnc vs UNet)')
print('  plot_metric_curves.png    — Val Dice & IoU over epochs')
print('  plot_loss_breakdown.png   — Stacked loss term breakdown')
print('  plot_metric_bars.png      — Grouped bar chart (5 metrics)')
print('  plot_radar.png            — Radar / spider chart (5 metrics)')
print('  plot_visual_grid.png      — Side-by-side prediction grid')
print('  plot_threshold_sweep.png  — Dice vs decision threshold')